In [6]:
# Import or install Sionna
try:
    import sionna.rt
except ImportError as e:
    import os
    os.system("pip install sionna-rt")
    import sionna.rt

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

no_preview = True  # Disable the interactive preview widget.

# Import relevant components from Sionna RT
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, Camera,\
                      PathSolver, RadioMapSolver, subcarrier_frequencies

from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, RadioMaterial, Camera
from sionna.rt.utils import r_hat

In [ ]:
import importlib
import sys

def check_tensorflow():
    if importlib.util.find_spec("tensorflow") is None:
        print("[TensorFlow]  not installed")
        return
    import tensorflow as tf
    print(f"[TensorFlow] version: {tf.__version__}")
    print(f"[TensorFlow] CUDA support: {tf.test.is_built_with_cuda()}")
    print(f"[TensorFlow] available GPUs: {tf.config.list_physical_devices('GPU')}")

def check_pytorch():
    if importlib.util.find_spec("torch") is None:
        print("\n[PyTorch]  not installed")
        return
    import torch
    print(f"\n[PyTorch] version: {torch.__version__}")
    print(f"[PyTorch] CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"[PyTorch] GPU count: {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            print(f"[PyTorch] GPU {i} name: {torch.cuda.get_device_name(i)}")

def check_sionna():
    if importlib.util.find_spec("sionna") is None:
        print("\n[Sionna]  not installed")
        return
    import sionna
    print(f"\n[Sionna] version: {sionna.__version__}")
    try:
        # Check whether Sionna can access TensorFlow GPUs.
        import tensorflow as tf
        gpus = tf.config.list_physical_devices('GPU')
        if gpus:
            print(f"[Sionna] available GPU count: {len(gpus)}")
        else:
            print("[Sionna] no GPU detected through TensorFlow")
    except Exception as e:
        print(f"[Sionna] GPU check failed: {e}")

if __name__ == "__main__":
    print(f"Python version: {sys.version}\n")
    check_tensorflow()
    check_pytorch()
    check_sionna()


Python version: 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]

[TensorFlow] version: 2.19.0
[TensorFlow] CUDA support: True
[TensorFlow] available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

[PyTorch] version: 2.5.1+cu121
[PyTorch] CUDA available: True
[PyTorch] GPU count: 1
[PyTorch] GPU 0 name: NVIDIA GeForce RTX 3060 Ti

[Sionna] version: 1.1.0
[Sionna] available GPU count: 1


In [8]:
import tensorflow as tf
print("GPU devices:", tf.config.list_physical_devices('GPU'))

GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [9]:
import os
gpu_num = 0 # Use "" to use the CPU
os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import shapely
from utils import *
from pyproj import Transformer

import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import mitsuba as mi
import csv

In [10]:
# Load integrated scene
# sionna.rt.scene.munich
from pathlib import Path

def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "assets" / "raytrack_scene" / "scene.xml").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the RayLoc project root.")

PROJECT_ROOT = find_project_root()
GENERATED_DIR = PROJECT_ROOT / "generated"
GENERATED_DIR.mkdir(parents=True, exist_ok=True)
SCENE_PATH = PROJECT_ROOT / "assets" / "raytrack_scene" / "scene.xml"

scene = load_scene(str(SCENE_PATH))
# scene.preview()

2026-07-08 04:11:54 WARN wrk8 [HDRFilm] Monochrome mode enabled, setting film output pixel format to 'luminance' (was rgb).

jit_optix_api_init(): could not find symbol optixQueryFunctionTable


RuntimeError: ​[xml.cpp:1078] Error while loading "<string>" (near line 1, col 1): could not instantiate scene plugin of type "scene": Could not initialize OptiX!

In [ ]:
#convert lat lon to coordinate
def sionna_coord(target_latlon):

    #bound
    bl = (-122.312299126737, 47.65074792715546)
    ur = (-122.30187970485449, 47.65690207846001)
    box = [bl, (bl[0], ur[1]), ur, (ur[0], bl[1])]

    target_coord_flipped = (target_latlon[1], target_latlon[0])

    #find epsg code for projection
    EPSG_code = get_utm_epsg_code_from_gps(bl[0], bl[1])
    to_utm = Transformer.from_crs("EPSG:4326", EPSG_code, always_xy=True)

    target_utm = to_utm.transform(target_coord_flipped[0], target_coord_flipped[1])

    #find center
    ground_utm = [to_utm.transform(lon, lat) for lon, lat in box]
    ground_polygon_utm = shapely.geometry.Polygon(ground_utm)
    ground_envelop = ground_polygon_utm.envelope
    center_x = ground_envelop.centroid.x
    center_y = ground_envelop.centroid.y

    #relative coord
    relative_coord = (target_utm[0] - center_x, target_utm[1] - center_y)

    return relative_coord

def sionna_coord_batch(latlon_array):
    """
    latlon_array: [N,2] -> [[lat, lon], ...]
    return: [N,2] -> [[x, y], ...]
    """

    latlon_array = np.asarray(latlon_array)
    assert latlon_array.shape[1] == 2, "Input must be Nx2 [lat, lon]"

    # =========================
    # 1. bounding box
    # =========================
    bl = (-122.312299126737, 47.65074792715546)
    ur = (-122.30187970485449, 47.65690207846001)
    box = [bl, (bl[0], ur[1]), ur, (ur[0], bl[1])]

    # =========================
    # 2. projection (computed once)
    # =========================
    EPSG_code = get_utm_epsg_code_from_gps(bl[0], bl[1])
    to_utm = Transformer.from_crs("EPSG:4326", EPSG_code, always_xy=True)

    # =========================
    # 3. compute center
    # =========================
    ground_utm = np.array([to_utm.transform(lon, lat) for lon, lat in box])
    ground_polygon_utm = shapely.geometry.Polygon(ground_utm)
    center_x = ground_polygon_utm.envelope.centroid.x
    center_y = ground_polygon_utm.envelope.centroid.y

    # =========================
    # 4. lat/lon -> utm (vectorized)
    # =========================
    lats = latlon_array[:, 0]
    lons = latlon_array[:, 1]

    # pyproj supports vectorized transforms.
    utm_x, utm_y = to_utm.transform(lons, lats)

    utm_x = np.array(utm_x)
    utm_y = np.array(utm_y)

    # =========================
    # 5. relative coord
    # =========================
    rel_x = utm_x - center_x
    rel_y = utm_y - center_y

    return np.stack([rel_x, rel_y], axis=1)

In [ ]:
#calibration test
target = np.array([47.655107665483904, -122.3086761875066])

relative_coord = sionna_coord(target)

# set tx
scene.remove("tx")

tx = Transmitter(name="tx",
                 position=[relative_coord[0], relative_coord[1], 0],
                 orientation=[0,0,0])

scene.add(tx)

#scene.preview()

In [ ]:
#configure building material
#def material itu concrete
# Sionna radio-material reference: https://nvlabs.github.io/sionna/api/rt.html#radio-materials
def medium_dry_ground_callback(f_hz):
   a = 15
   b = -0.1
   c = 0.035
   d = 1.63

   relative_permittivity = a * (f_hz*10e-9)**b
   conductivity = c * (f_hz*10e-9)**d
   return (relative_permittivity, conductivity)

def concrete_callback(f_hz):
   # a = 5.24 #5.24
   # # a = 4.65
   # b = 0
   # c = 0.0462
   # d = 0.7822
   a = 1.48 #5.24
   # a = 4.65
   b = 0
   c = 0.0011
   d = 1.0750

   relative_permittivity = a * (f_hz*10e-9)**b
   conductivity = c * (f_hz*10e-9)**d
   return (relative_permittivity, conductivity)

concrete_all_frequency = RadioMaterial("concrete_all_frequency",
                                frequency_update_callback=concrete_callback)
scene.add(concrete_all_frequency)

ground_all_frequency = RadioMaterial("ground_all_frequency",
                                frequency_update_callback=medium_dry_ground_callback)
scene.add(ground_all_frequency)

In [ ]:
print(list(scene.objects))

['no-name-4', 'ground']


In [ ]:
for i in range(len(list(scene.objects.keys()))):
    obj_name = list(scene.objects.keys())[i]
    obj = scene.get(obj_name)

    if (obj_name != "ground"):
        obj.radio_material = concrete_all_frequency
    else:
        obj.radio_material = ground_all_frequency

    obj.radio_material.scattering_coefficient = 0.5

scene.remove('itu_concrete')
scene.remove('itu_medium_dry_ground')

#frequency, tx rx array, and material configuration
scene.frequency = 915e6 # in Hz; implicitly updates RadioMaterials
# scene.frequency = 2e9

scene.synthetic_array = True # If set to False, ray tracing will be done per antenna element (slower for large arrays)

#print(scene.objects)
# obj = scene.get(list(scene.objects.keys())[0])
# print(list(scene.objects.keys()))
# obj.radio_material = concrete_all_frequency
# obj.radio_material.scattering_coefficient = 0.5

scene.tx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="dipole",
                             polarization="V")

scene.rx_array = scene.tx_array

In [ ]:
# =========================
# User settings
# =========================
samples_per_tx = int(1e8)
rx_height_list = [0.9, 1.2, 1.5, 1.8, 2.1]

base_save_dir = GENERATED_DIR / "rx_height_sensitivity_results"
os.makedirs(base_save_dir, exist_ok=True)

# ----- TX location -----
lab1_gps = (47.65361769298625, -122.30694886733762)   # Transmitter anchor.
lab1_coord = sionna_coord(lab1_gps)

# Fixed transmitter height.
tx_height = 13.0

eps = 1e-30

def to_numpy(x):
    if isinstance(x, np.ndarray):
        return x
    if hasattr(x, "numpy"):
        return x.numpy()
    return np.asarray(x)

def lin2db(x, eps=1e-30):
    x = np.asarray(x, dtype=np.float64)
    return 10.0 * np.log10(np.maximum(x, eps))

# =========================
# Fix TX once
# =========================
try:
    scene.remove("tx")
except Exception:
    pass

tx = Transmitter(
    name="tx",
    position=[lab1_coord[0], lab1_coord[1], tx_height],
    orientation=[0, 0, 0]
)
scene.add(tx)

rm_solver = RadioMapSolver()

# ==========================================================
# Step 1: run one default map to infer the default XY extent
# Default radio map is at z = 1.5m if center is not provided
# ==========================================================
print("Running one baseline map to infer default XY measurement area...")

rm_ref = rm_solver(
    scene=scene,
    max_depth=5,
    refraction=True,
    cell_size=(1.0, 1.0),
    samples_per_tx=samples_per_tx,
    specular_reflection=True,
    diffuse_reflection=True,
)

cc_ref = to_numpy(rm_ref.cell_centers)   # expected shape [H, W, 3]
x_coords = cc_ref[..., 0]
y_coords = cc_ref[..., 1]

x_min, x_max = np.min(x_coords), np.max(x_coords)
y_min, y_max = np.min(y_coords), np.max(y_coords)

center_x = 0.5 * (x_min + x_max)
center_y = 0.5 * (y_min + y_max)

# size should span the whole inferred plane
size_x = (x_max - x_min) + 1.0
size_y = (y_max - y_min) + 1.0

print(f"Inferred map center XY = ({center_x:.3f}, {center_y:.3f})")
print(f"Inferred map size      = ({size_x:.3f}, {size_y:.3f})")

summary_rows = []

# =========================
# Sweep RX heights
# =========================
for rx_height in rx_height_list:
    print("\n" + "=" * 60)
    print(f"Running RX height = {rx_height} m")
    print("=" * 60)

    save_dir = os.path.join(base_save_dir, f"rx_h_{rx_height:g}m")
    os.makedirs(save_dir, exist_ok=True)

    t0 = time.time()

    # Explicitly set measurement plane height by center.z = rx_height
    center_pt = mi.Point3f(
        float(center_x),
        float(center_y),
        float(rx_height)
    )

    orientation_pt = mi.Point3f(
        0.0, 0.0, 0.0
    )

    size_pt = mi.Point2f(
        float(size_x),
        float(size_y)
    )

    rm = rm_solver(
        scene=scene,
        center=center_pt,
        orientation=orientation_pt,
        size=size_pt,
        cell_size=mi.Point2f(1.0, 1.0),
        max_depth=5,
        refraction=True,
        samples_per_tx=samples_per_tx,
        specular_reflection=True,
        diffuse_reflection=True,
    )

    rss = np.squeeze(to_numpy(rm.rss)).astype(np.float64)
    pg  = np.squeeze(to_numpy(rm.path_gain)).astype(np.float64)
    cc  = to_numpy(rm.cell_centers)

    # ----- dB -----
    valid_mask = rss > 0
    rss_db = np.full_like(rss, np.nan, dtype=np.float64)
    rss_db[valid_mask] = 10.0 * np.log10(rss[valid_mask])

    rss_mean_db = lin2db(rss, eps)
    pg_mean_db  = lin2db(pg, eps)

    # ----- summary metrics -----
    coverage_ratio = float(np.mean(valid_mask))
    rss_global_mean = float(np.mean(rss))
    rss_global_std  = float(np.std(rss))
    pg_global_mean  = float(np.mean(pg))
    pg_global_std   = float(np.std(pg))

    if np.any(valid_mask):
        rss_global_mean_db = float(np.nanmean(rss_db))
        rss_global_std_db  = float(np.nanstd(rss_db))
    else:
        rss_global_mean_db = np.nan
        rss_global_std_db  = np.nan

    # -------------------------
    # Save arrays
    # -------------------------
    np.savez_compressed(
        os.path.join(save_dir, "rm_stats.npz"),
        rss_map=rss,
        pg_map=pg,
        rss_db=rss_db,
        rss_mean_db=rss_mean_db,
        pg_mean_db=pg_mean_db,
        cell_centers=cc,
        valid_mask=valid_mask.astype(np.uint8),
    )

    meta = {
        "tx_height": float(tx_height),
        "rx_height": float(rx_height),
        "samples_per_tx": int(samples_per_tx),
        "tx_position": [float(lab1_coord[0]), float(lab1_coord[1]), float(tx_height)],
        "rm_center": [float(center_x), float(center_y), float(rx_height)],
        "rm_size": [float(size_x), float(size_y)],
        "coverage_ratio": coverage_ratio,
        "rss_global_mean": rss_global_mean,
        "rss_global_std": rss_global_std,
        "pg_global_mean": pg_global_mean,
        "pg_global_std": pg_global_std,
        "rss_global_mean_db": rss_global_mean_db,
        "rss_global_std_db": rss_global_std_db,
        "elapsed_sec": time.time() - t0,
    }

    with open(os.path.join(save_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    summary_rows.append(meta)

    # -------------------------
    # Save one figure for each RX height
    # -------------------------
    plt.figure(figsize=(7, 5))

    # extent for correct XY axis labels
    x_min_plot = np.nanmin(cc[..., 0])
    x_max_plot = np.nanmax(cc[..., 0])
    y_min_plot = np.nanmin(cc[..., 1])
    y_max_plot = np.nanmax(cc[..., 1])

    im = plt.imshow(
        rss_db,
        origin="lower",
        extent=[x_min_plot, x_max_plot, y_min_plot, y_max_plot],
        aspect="equal"
    )
    plt.colorbar(im, label="RSS (dB)")
    plt.scatter(lab1_coord[0], lab1_coord[1], marker="*", s=120, label="TX")
    plt.xlabel("x [m]")
    plt.ylabel("y [m]")
    plt.title(f"RSS map at RX height = {rx_height} m")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "rss_db_map.png"), dpi=200)
    plt.close()

# =========================
# Save summary
# =========================
summary_csv_path = os.path.join(base_save_dir, "rx_height_summary.csv")
fieldnames = [
    "tx_position", "tx_height", "rx_height", "samples_per_tx",
    "rm_center", "rm_size",
    "coverage_ratio",
    "rss_global_mean", "rss_global_std",
    "pg_global_mean", "pg_global_std",
    "rss_global_mean_db", "rss_global_std_db",
    "elapsed_sec"
]

with open(summary_csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(summary_rows)

with open(os.path.join(base_save_dir, "rx_height_summary.json"), "w") as f:
    json.dump(summary_rows, f, indent=2)

print(f"\nAll RX heights finished. Summary saved to: {summary_csv_path}")

Running one baseline map to infer default XY measurement area...
Inferred map center XY = (0.109, 0.283)
Inferred map size      = (1183.000, 1037.000)

Running RX height = 0.9 m

Running RX height = 1.2 m

Running RX height = 1.5 m

Running RX height = 1.8 m

Running RX height = 2.1 m

All RX heights finished. Summary saved to: /mnt/c/Users/NEWTLAB/Documents/RayLoc/rx_height_sensitivity_results/rx_height_summary.csv
